In [13]:
import fine.IOManagement.xarrayIO as xrIO

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# How to save an energy system model instance and set it back up? 

**Xarray and NetCDF files to the rescue!** The data contained within an Energy System Model (ESM) instance and the optimization results is vast and complex. Saving it directly is not possible. It can, however, be saved as a NetCDF file which supports complex data structures. 

#### What exactly is NetCDF? 
NetCDF (Network Common Data Format) is a set of software libraries and machine-independent data formats that support the creation, access, and sharing of array-oriented scientific data. It is also a community standard for sharing scientific data. 

#### Python modules that support working with NetCDF files:
1. netcdf4-python: Official Python interface to netCDF files
2. PyNIO: To access different file formats such as netCDF, HDF, and GRIB
3. xarray: Based on NumPy and pandas

Note: xarray module is used here. 

For our use case, the following functionalities are provided: 
* Conversion of ESM instance to xarray dataset. Additionally, possible to save this dataset as NetCDF file in a desired folder, with a desired file name. 
* Conversion of xarray dataset/saved NetCDF file back to ESM instance.

#### High-level structure of the data: 

<img src="figures/overall_structure.PNG" style="width: 1000px;"/>


#### Structure of xarray dataset - For a non-transmission component: 

<img src="figures/non_transmission.PNG" style="width: 1000px;"/>

#### Structure of xarray dataset - For a transmission component: 

<img src="figures/transmission.PNG" style="width: 1000px;"/>


## Conversion of ESM instance to xarray dataset and saving it as a NetCDF file

### STEP 1. Set up your  ESM instance 

In [14]:
import fine as fn
from getModel import getModel

esM = getModel()
esM.optimize(solver=fn.utils.ImplementedSolvers.STANDARD_SOLVER.value)

Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Rocky Linux 9.5 (Blue Onyx)")

CPU model: Intel(R) Xeon(R) Gold 6144 CPU @ 3.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 3 threads

Non-default parameters:
TSPort  41955
QCPDual  1
Threads  3

Optimize a model with 96 rows, 74 columns and 240 nonzeros (Min)
Model fingerprint: 0x84519374
Model has 10 linear objective coefficients
Coefficient statistics:
  Matrix range     [3e-01, 2e+03]
  Objective range  [7e-03, 9e+01]
  Bounds range     [1e+07, 2e+09]
  RHS range        [0e+00, 0e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Presolve removed 51 rows and 40 columns
Presolve time: 0.00s
Presolved: 45 rows, 34 columns, 177 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Tim

### STEP 2. Conversion to xarray datasets and saving as NetCDF file
You can convert the esM to xarray datasets with `esm_to_datasets` and access Input, Parameters or Result.


In [15]:
esm_datasets = xrIO.writeEnergySystemModelToDatasets(esM)

In [16]:
esm_datasets["Input"]["Sink"]["Industry site"][
    "ts_operationRateFix"
].to_dataframe().unstack()

ts_operationRateFix                 
space ElectrolyzerLocation IndustryLocation
time                                       
0                      0.0       13140000.0
1                      0.0       13140000.0
2                      0.0       13140000.0
3                      0.0       13140000.0

In [17]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"]

<xarray.Dataset> Size: 544B
Dimensions:                          (location: 2, time: 4)
Coordinates:
  * time                             (time) int64 32B 0 1 2 3
  * location                         (location) <U20 160B 'ElectrolyzerLocati...
Data variables: (12/19)
    NPVcontribution                  (location) float64 16B 1.52e+06 0.0
    TAC                              (location) float64 16B 1.52e+06 0.0
    capacity                         (location) float64 16B nan nan
    capexCap                         (location) float64 16B nan nan
    capexIfBuilt                     (location) float64 16B nan nan
    commissioning                    (location) float64 16B nan nan
    ...                               ...
    operation_annual                 (location) float64 16B 7.509e+07 nan
    opexCap                          (location) float64 16B nan nan
    opexIfBuilt                      (location) float64 16B nan nan
    opexOp                           (location) float64 16B 0.0 nan
    revenueLifetimeShorteningResale  (location) float64 16B nan nan
    operationTimeSeriesOptimum       (time, location) float64 64B 1.877e+07 ....
Attributes: (12/18)
    NPVcontribution:                  [1 Euro]
    TAC:                              [1 Euro/a]
    capacity:                         [kW$_{el}$]
    capexCap:                         [1 Euro/a]
    capexIfBuilt:                     [1 Euro/a]
    commissioning:                    [kW$_{el}$]
    ...                               ...
    operation:                        [kW$_{el}$*h]
    operation_annual:                 [kW$_{el}$*h/a]
    opexCap:                          [1 Euro/a]
    opexIfBuilt:                      [1 Euro/a]
    opexOp:                           [1 Euro/a]
    revenueLifetimeShorteningResale:  [1 Euro]

In [18]:
esm_datasets["Parameters"]

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Attributes: (12/15)
    locations:                  {'ElectrolyzerLocation', 'IndustryLocation'}
    commodities:                {'hydrogen', 'electricity'}
    commodityUnitsDict:         {'electricity': 'kW$_{el}$', 'hydrogen': 'kW$...
    numberOfTimeSteps:          4
    hoursPerTimeStep:           2190
    startYear:                  0
    ...                         ...
    costUnit:                   1 Euro
    lengthUnit:                 km
    verboseLogLevel:            1
    balanceLimit:               None
    pathwayBalanceLimit:        None
    annuityPerpetuity:          False

Or save it directly to NetCDF with `esm_to_netcdf`:

In [19]:
_ = xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath="my_esm.nc", overwriteExisting=True
)

### STEP 3. Load esM from NetCDF file or xarray datasets

You can load an esM from file with `netcdf_to_esm`.

In [20]:
esm_from_netcdf = xrIO.readNetCDFtoEnergySystemModel("my_esm.nc")

In [21]:
esm_from_netcdf.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


Or from datasets with `datasets_to_esm`.

In [22]:
esm_from_datasets = xrIO.convertDatasetsToEnergySystemModel(esm_datasets)

In [23]:
esm_from_datasets.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


In [ ]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"][
    "operationTimeSeries"
]

<xarray.DataArray 'operationTimeSeriesOptimum' (time: 4, location: 2)> Size: 64B
array([[18771428.57142857,               nan],
       [37542857.14285714,               nan],
       [       0.        ,               nan],
       [18771428.57142857,               nan]])
Coordinates:
  * time      (time) int64 32B 0 1 2 3
  * location  (location) <U20 160B 'ElectrolyzerLocation' 'IndustryLocation'